In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# c01 时间到 DCT2＋CDC32 同链路读取
Run all 仅使用既有 c01 receiver/received 下 FULL、CROP、DELETE1、REPEAT1、DELETE3 的二代 H264 received.mp4 和已保存 V3 submission。不会重新物化、重算时间接口或使用旧 M3/e01 缓存。默认路径已填写；源码普通下载，实际依赖导入修复，无摘要、固定版本或 Git 状态门禁。此源已见，仅为 same-chain development。

先冻结全部 B/W/U 引用，再共享 60 个 E41。B 使用明确最终时间输出；W 对 B 的完整局部计划循环加 11 cell，同实际引用、无支持和失败均保留；U 不读时间，以收到时钟和固定 p0 读取。四帧 CELL_GEOMETRY_FULL 与完整41帧上下文风险分别记录；未知/跳变不会扩大为整41硬拒绝。DCT2、CDC32冲突擦除及技术缺失语义不变，各臂分别聚合，每个攻击视图单独报告。

一次固定256逻辑步骤：plan5、probe5、decode5、VAE load1、E4160、DCT project60、keyed decode60、local read45、aggregate15。codec共10，encode0；9600原始身份位、76800票槽、1440局部槽、480最终槽（每臂160）保留。RSS8GiB从计划读取覆盖至盲终态保存；codec每次300秒，load900秒，E41每次600秒/累计7200秒，CPU接收保存1800秒（含最终盲终态保存）。模型/E41耗时与保存超限在返回后检查，不承诺调用中硬中断，实测另记。GPU型号不是门禁，不暗改dtype/窗口/模型重试。

第一次真实调用前保存attempt、完整账本与固定槽骨架。任何失败或中断消费这次；再次Run all只读，不补领未完成步骤。即使硬终止未保存blind_terminal，也依据已保存操作回执展示480/1440槽及raw60单元；缺失/未完成处为UNRESOLVED_RECEIPT，不打开位缓存或真值补算。所有盲槽终态保存后才打开后评mapping和原同源固定消息。输出在OUTPUT的plans.json、raw_bank.json、local、final、blind_terminal.json、evaluation.json；原失败和未读保留。

默认仅打印有界状态摘要、分母、操作计数、资源、已有评价状态和输出路径，缺失字段显示None；完整result变量和磁盘逐槽记录保留，不展开全部refs，不在显示时重新后评或评分。

本地工程交付未执行真实媒体/模型。未来读回不能覆盖原native目标失败/最优性未证，不能声称QIM对身份无损、独立新源确认、Ø/FPR、物理身份或完整同步。共同输出BER、覆盖增减和错误归属分开；全擦除不通过。W的定义性错组本身不证明同步收益；字面位同好而组标签不同不算改善解码。四组8bit在组内重复，不能区分同组cell或任意长位置。


In [ ]:
SOURCE = '/content/drive/MyDrive/Video-WM/B-Conditional-Time-Validation-V1/fixed-c01-v1'
OUTPUT = '/content/drive/MyDrive/Video-WM/B-Time-Payload-CDC32-V1/fixed-c01-v1'
SOURCE_DIRECTORY = None
SOURCE_REF = 'dev/local-joint-state-payload-v1'
MODEL_ID = None


In [ ]:
from pathlib import Path
import importlib.util, json, sys, tempfile, urllib.request
if (Path(OUTPUT)/'attempt.json').exists():
    result = {'status': 'READ_ONLY_EXISTING_ATTEMPT', 'physical_calls_this_invocation': 0, 'saved': {}, 'read_errors': {}}
    for name in ('attempt', 'state', 'counts', 'resources', 'evaluation', 'result'):
        try:
            saved = json.loads((Path(OUTPUT)/(name+'.json')).read_text())
            if not isinstance(saved, dict): raise ValueError('saved receipt object required')
            result['saved'][name] = saved
        except (OSError, ValueError) as error: result['read_errors'][name] = str(error)
    # Record-only display uses the fixed skeleton saved before the first operation.
    # No source, planner, bit cache, truth, model or evaluator is loaded here.
    import copy
    result['recovered_artifacts'] = {}
    state = result['saved'].get('state', {})
    for item in result['saved'].get('attempt', {}).get('recovery_layout', []):
        value, error, unconfirmed = copy.deepcopy(item['fallback']), None, None
        try:
            path = (Path(OUTPUT)/item['path']).resolve()
            if not path.is_relative_to(Path(OUTPUT).resolve()): raise ValueError('receipt path outside output')
            saved = json.loads(path.read_text())
            if item['kind'] == 'raw_bank':
                if not isinstance(saved, dict): raise ValueError('raw metadata object required')
                for key in value:
                    fixed, row = value[key], saved.get(key)
                    statuses = [state[op].get('status') if isinstance(state.get(op), dict) else None for op in fixed['operations']]
                    known = 'RETURNED' if statuses[-1] == 'RETURNED' else next((s for s in reversed(statuses) if s in ('FAILED', 'INTERRUPTED', 'RESOURCE_UNRESOLVED')), None)
                    if known is None: known = 'DEPENDENCY_UNRESOLVED' if all(s in ('NOT_STARTED', 'DEPENDENCY_UNRESOLVED') for s in statuses) else 'UNRESOLVED_RECEIPT'
                    valid = isinstance(row, dict) and known != 'UNRESOLVED_RECEIPT' and row.get('status') == known and all(row.get(k) == fixed[k] for k in ('raw_bit_slots', 'vote_slots', 'encoder_input_support'))
                    if valid: value[key] = dict(row, **{k: fixed[k] for k in ('raw_bit_slots', 'vote_slots', 'encoder_input_support', 'operations')})
                    else:
                        fixed.update(operation_status=known, unconfirmed_raw_metadata=row)
                        error = 'raw metadata incomplete or unconfirmed; fixed unit fields retained'
            elif not isinstance(state.get(item['operation']), dict) or state[item['operation']].get('status') != 'RETURNED':
                unconfirmed, error = saved, 'operation receipt not returned; saved artifact retained as unconfirmed'
            else:
                rows = saved.get('slots') if isinstance(saved, dict) else None
                if not isinstance(rows, list): raise ValueError('slot receipt list required')
                for i, row in enumerate(rows[:32]):
                    if isinstance(row, dict) and type(row.get('slot')) is int and row['slot'] == i and 'value' in row and (row['value'] is None or type(row['value']) is int and row['value'] in (0, 1)) and isinstance(row.get('status'), str): value['slots'][i] = row
                    else: error = 'invalid slot record retained as unresolved'
                if len(rows) != 32: error = 'incomplete slot receipt; fixed32 retained'
        except (OSError, ValueError, TypeError) as exc: error = str(exc)
        result['recovered_artifacts'][item['path']] = {'kind': item['kind'], 'value': value, 'error': error, 'unconfirmed_artifact': unconfirmed}
    if not result['recovered_artifacts']: result['read_errors']['fixed_slots'] = 'saved attempt skeleton unreadable; use the standalone read-only CLI to reconstruct fixed unresolved slots'
elif Path(OUTPUT).exists() and (not Path(OUTPUT).is_dir() or any(Path(OUTPUT).iterdir())):
    result = {'status': 'OUTPUT_CONFLICT', 'physical_calls_this_invocation': 0}
else:
    try:
        with tempfile.TemporaryDirectory(prefix='time-payload-bootstrap-') as temporary:
            root = Path(SOURCE_DIRECTORY) if SOURCE_DIRECTORY is not None else Path(temporary)
            if SOURCE_DIRECTORY is None:
                files = ('scripts/b_time_payload_cdc32_v1_entry.py', 'scripts/b_conditional_time_output_v1_entry.py',
                         'scripts/b_framewise_qim_v1_entry.py', 'experiments/wan_state_clock/b_time_payload_cdc32_v1_records.py',
                         'experiments/wan_state_clock/b_minimal_l_v1_records.py', 'experiments/wan_state_clock/configs/b_time_payload_cdc32_v1.json')
                for relative in files:
                    path = root/relative; path.parent.mkdir(parents=True, exist_ok=True)
                    urllib.request.urlretrieve('https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/'+SOURCE_REF+'/'+relative, path)
            spec = importlib.util.spec_from_file_location('time_payload_entry', root/'scripts/b_time_payload_cdc32_v1_entry.py')
            entry = importlib.util.module_from_spec(spec)
            previous = sys.dont_write_bytecode; sys.dont_write_bytecode = True
            try: spec.loader.exec_module(entry)
            finally: sys.dont_write_bytecode = previous
            result = entry.launch(SOURCE, OUTPUT, SOURCE_DIRECTORY, SOURCE_REF, MODEL_ID)
    except Exception as error:
        result = {'status': 'PREPARATION_FAILED', 'error': str(error), 'physical_calls_this_invocation': 0}
# Display saved metadata only; retain the complete result variable and files.
saved_summary = result.get('saved', {})
for name in ('counts', 'resources', 'result'):
    if name not in saved_summary:
        try:
            value = json.loads((Path(OUTPUT)/(name+'.json')).read_text())
            if isinstance(value, dict): saved_summary[name] = value
        except (OSError, ValueError): pass
counts = saved_summary.get('counts', {})
resources = saved_summary.get('resources', {})
den = result.get('denominators', saved_summary.get('attempt', {}).get('denominators', saved_summary.get('result', {}).get('denominators', {})))
errors = result.get('errors', saved_summary.get('result', {}).get('errors', result.get('read_errors', {})))
recovery = result.get('recovered_artifacts', {})
summary = {'status': result.get('status'), 'output': str(OUTPUT),
           'evaluation_status': result.get('evaluation_status', saved_summary.get('result', {}).get('evaluation_status')),
           'denominators': {k: den.get(k) if type(den.get(k)) is int else None for k in ('logical_operations', 'E41', 'codec', 'raw_identity_bit_slots', 'coefficient_vote_slots', 'local_slots', 'final_slots', 'per_arm_final_slots')} if isinstance(den, dict) else None,
           'operation_counts': {str(k)[:64]: {f: v.get(f) if type(v.get(f)) is int else None for f in ('planned', 'started', 'returned')} for k, v in list(counts.items())[:32] if isinstance(v, dict)} if isinstance(counts, dict) and counts else None,
           'resources': {k: resources.get(k) if type(resources.get(k)) in (int, float) else resources[k][:160] if isinstance(resources.get(k), str) else None for k in ('peak_resident_bytes', 'peak_cuda_allocated_bytes', 'cpu_receive_save_seconds', 'E41_seconds', 'measurement_end')} if isinstance(resources, dict) and resources else None,
           'receipt_recovery': {'records': len(recovery), 'unconfirmed_records': sum(bool(v.get('error')) for v in recovery.values())} if recovery else None,
           'errors': {str(k)[:80]: str(v)[:240] for k, v in list(errors.items())[:8]} if isinstance(errors, dict) else None,
           'full_records': 'Complete result retained in result; fixed slot records and refs remain in OUTPUT.'}
print(summary)
